# Evaluating Custom Criteria with G-Eval

Most metrics (faithfulness, tool correctness, ...) check one specific thing. But often you need to
grade something subjective and specific to your app — "is this on-brand?", "is this medically
careful?", "did it actually solve the user's problem?". **G-Eval** lets you describe that criteria
in plain English and uses an LLM judge, guided by chain-of-thought reasoning, to score it
consistently.

| Step | What happens |
| --- | --- |
| 1. Step generation | Your `criteria` (or explicit `evaluation_steps`) becomes a short chain-of-thought checklist |
| 2. Prompt creation | The checklist + your test case fields are assembled into one judge prompt |
| 3. Scoring | The judge LLM outputs a 1-5 score |
| 4. Normalization | DeepEval converts that into a 0-1 score using the judge's token probabilities |

## 1. Install packages

Just `deepeval` and `google-genai` (so DeepEval's judge can talk to Gemini). No model-under-test
call in this notebook, so no Groq needed.

In [1]:
%pip install -q deepeval google-genai

/Users/yashpatil/Developer/AI/KNA/Udemy_live/Evals_with_DeepEvals/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Add your Gemini API key

G-Eval needs an LLM judge. We use Gemini so you don't need an OpenAI key — get a free one at
https://aistudio.google.com/apikey. Entered with `getpass` so it's never printed or saved into the
notebook file.

In [1]:
import os, getpass

os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")  # skip DeepEval's anonymous telemetry

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]  # some libs read this name instead

print("Key set.")

Key set.


## 3. A couple of example outputs to evaluate

To keep this notebook focused on the *evaluator* (not on building an app), we hand-write the
outputs instead of calling a model to generate them. One answer is good, one is flawed on purpose
— watch how G-Eval scores each.

In [2]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "Happy to help! We offer full refunds within 30 days of purchase, no questions asked. "
            "Just reply here with your order number and I'll process it."
        ),
        expected_output="Refunds are available within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="No refunds. Read the policy page next time.",
        expected_output="Refunds are available within 30 days of purchase.",
    ),
]

## The judge: `GeminiModel`

Every DeepEval metric defaults to OpenAI unless you pass `model=`. We build **one Gemini judge**
here and hand it to both metrics below.

In [3]:
from deepeval.models import GeminiModel

judge = GeminiModel(model="gemini-3.5-flash", api_key=os.environ["GEMINI_API_KEY"], temperature=0)
print("Judge ready:", judge.get_model_name())

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Judge ready: gemini-3.5-flash (Gemini)


## 4. Define two G-Eval metrics

Two ways to tell G-Eval what to check:

- **`criteria`** — plain-English description; DeepEval turns it into evaluation steps for you.
- **`evaluation_steps`** — you write the checklist yourself, which skips that generation step and
  makes scoring more consistent run to run.

In [4]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

correctness = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct given the expected output.",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model=judge,
)

tone = GEval(
    name="Professional Tone",
    evaluation_steps=[
        "Check whether the response is polite and professional.",
        "Penalize sarcasm, rudeness, or dismissive language.",
        "Reward clear, respectful phrasing even if brief.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=judge,
)

## 5. Run the evaluation

Only 2 test cases x 2 metrics = 4 judge calls, and `max_concurrent=1` keeps us well under Gemini's
free-tier rate limit.

In [5]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=[correctness, tone],
    async_config=AsyncConfig(max_concurrent=1),          # be gentle on Gemini free-tier rate limits
    display_config=DisplayConfig(print_results=False),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),
)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gemini-3.5-flash (Gemini), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Professional Tone [GEval] Metric! (using gemini-3.5-flash (Gemini), 
strict=False, async_mode=True)...

/Users/yashpatil/Developer/AI/KNA/Udemy_live/Evals_with_DeepEvals/.venv/lib/python3.12/site-packages/rich/live.py:2
60: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

...

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.

⚠ WARNING: No hyperparameters logged.
» ]8;id=9264725;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 91.78s | token cost: 0.006268500000000001 USD)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Read the results

Each `TestResult` carries a `metrics_data` list — one entry per metric, with a `score`, a `reason`
(the judge's explanation), and whether it `success`-fully passed its threshold. If the judge call
itself fails (rate limit, transient error), `score` is `None` and the explanation lives in `.error`
instead of `.reason`.

In [6]:
for tr in results.test_results:
    print("=" * 90)
    print("ACTUAL OUTPUT:", tr.actual_output)
    for md_ in tr.metrics_data:
        verdict = "PASS" if md_.success else "FAIL"
        score = f"{md_.score:.2f}" if md_.score is not None else "ERRORED"
        print(f"  [{verdict}] {md_.name}: {score}")
        print(f"      reason: {md_.reason or md_.error}")

ACTUAL OUTPUT: Happy to help! We offer full refunds within 30 days of purchase, no questions asked. Just reply here with your order number and I'll process it.
  [PASS] Correctness [GEval]: 0.90
      reason: The actual output is fully aligned with the expected output, correctly stating that refunds are available within 30 days of purchase. It adds helpful conversational filler and instructions on how to initiate the refund, which do not contradict the core fact.
  [PASS] Professional Tone [GEval]: 1.00
      reason: The response is exceptionally polite, professional, and welcoming, starting with a friendly greeting and clearly explaining the 30-day refund policy with simple, actionable next steps.
ACTUAL OUTPUT: No refunds. Read the policy page next time.
  [FAIL] Correctness [GEval]: 0.00
      reason: The actual output directly contradicts the expected output by stating that there are no refunds, whereas the expected output explicitly allows refunds within 30 days of purchase. Addit

## Takeaway

G-Eval is the right tool whenever the thing you're grading is subjective or open-ended —
correctness, tone, coherence, safety, or a custom RAG rubric — because you describe it in plain
English instead of writing a scoring function. It's a poor fit for rigid, exact checks (e.g. "did
the output have exactly 3 bullet points") since it's still one probabilistic LLM call; for that,
reach for DeepEval's `DAG` metric instead. Two more knobs worth knowing about: `rubric=` to pin
specific score ranges to specific descriptions, and `strict_mode=True` to force a binary pass/fail.